# 07 — Model zoo: build every recommender

Each recommender is constructed **inline** here, one after another, and the whole set is saved to `artifacts/models.joblib`. **Evaluation lives in `08_evaluation.ipynb`**, which loads this file — so building and evaluating stay separate.

In [10]:
import glob, os, joblib
import numpy as np, pandas as pd
from book_recsys.data.splits import leave_last_n_out
from book_recsys.data.filters import filter_min_rating
from book_recsys.features.document import build_documents
from book_recsys.features.vectorize import tfidf_matrix
from book_recsys.models.classical.popularity import PopularityRecommender
from book_recsys.models.classical.svd import SvdRecommender
from book_recsys.models.content.content import ContentRecommender
from book_recsys.models.content.similar import SimilarItemsRecommender
from book_recsys.models.hybrid.learned import LearnedHybridRecommender

def find(name):
    for p in (f'artifacts/{name}', f'../artifacts/{name}'):
        if glob.glob(p): return glob.glob(p)[0]
    raise FileNotFoundError(name)

In [11]:
sample = pd.read_parquet(find('sample.parquet'))
catalog = pd.read_parquet(find('catalog.parquet'))
book_ids = catalog['book_id'].tolist()
emb = np.load(find('embeddings.npy'))
train, _ = leave_last_n_out(sample, n=1)   # same split eval holds out from
print(len(train), 'train interactions ·', len(book_ids), 'books')

12568544 train interactions · 468628 books


### Classical — popularity + collaborative filtering (SVD)

In [12]:
popularity = PopularityRecommender().fit(train)
svd = SvdRecommender(n_factors=64).fit(train)
svd_pos = SvdRecommender(n_factors=64).fit(filter_min_rating(train, 4))
print('popularity, svd, svd_rating>=4 fitted')

popularity, svd, svd_rating>=4 fitted


### Content-based — TF-IDF + embeddings

In [13]:
tfidf, _ = tfidf_matrix(build_documents(catalog), max_features=3000)
content_tfidf = ContentRecommender(book_ids, tfidf).fit()
content_emb = ContentRecommender(book_ids, emb).fit()
similar = SimilarItemsRecommender(book_ids, emb).fit()   # UC4 item-item
print('content_tfidf_full, content_emb, similar fitted')

content_tfidf_full, content_emb, similar fitted


### Hybrid — learned reranker over CF + content
Meta-model trained on a user sample (cost is one score_items per training user). Default `[cf, content]`; `+pop` is the popularity ablation.

In [14]:
users = train['user_id'].unique()
fit_users = np.random.default_rng(0).choice(users, size=min(5000, len(users)), replace=False)
hybrid_train = train[train['user_id'].isin(fit_users)]
hybrid = LearnedHybridRecommender({'cf': svd, 'content': content_emb}, seed=0).fit(hybrid_train)
hybrid_pop = LearnedHybridRecommender(
    {'cf': svd, 'content': content_emb, 'pop': popularity}, seed=0).fit(hybrid_train)
# ablation: same [cf, content] features but trained on popularity-weighted negatives
hybrid_popneg = LearnedHybridRecommender(
    {'cf': svd, 'content': content_emb}, seed=0, neg_sampling='popularity').fit(hybrid_train)
print('feature weights [cf, content]    :', hybrid.feature_weights())
print('feature weights [cf, content, pop]:', hybrid_pop.feature_weights())
print('feature weights [cf, content] popneg-train:', hybrid_popneg.feature_weights())

feature weights [cf, content]    : {'cf': 13.757231811702665, 'content': 0.5202612972505655}
feature weights [cf, content, pop]: {'cf': 9.469921904505552, 'content': 0.4959867659235521, 'pop': 1.0381615471955215}
feature weights [cf, content] popneg-train: {'cf': 0.8841069773169081, 'content': 0.4803329719579216}


### Persist the whole zoo for evaluation + the UI

In [15]:
models = {
    'popularity': popularity, 'svd': svd, 'svd_rating>=4': svd_pos,
    'content_tfidf_full': content_tfidf, 'content_emb': content_emb,
    'hybrid_cf_content': hybrid, 'hybrid_cf_content_pop': hybrid_pop,
    'hybrid_cf_content_popneg': hybrid_popneg,
    'similar': similar,
}
ART = os.path.dirname(find('sample.parquet'))   # same dir as the other artifacts
joblib.dump(models, f'{ART}/models.joblib')      # shared refs (svd) stored once
print('saved', list(models), '->', f'{ART}/models.joblib')

saved ['popularity', 'svd', 'svd_rating>=4', 'content_tfidf_full', 'content_emb', 'hybrid_cf_content', 'hybrid_cf_content_pop', 'hybrid_cf_content_popneg', 'similar'] -> ../artifacts/models.joblib


### Sanity — a sample recommendation from each model

In [16]:
id2title = dict(zip(catalog['book_id'], catalog['title']))
seed_book = train['book_id'].value_counts().index[0]
print('history seed:', id2title.get(seed_book, seed_book), '\n')
for name, rec in models.items():
    # 'similar' takes a single anchor book-id; the rest take a history list
    query = seed_book if name == 'similar' else [seed_book]
    print(f'{name}:')
    for b in rec.recommend(query, 5):
        print('   ', id2title.get(b, b))
    print()

history seed: Harry Potter and the Sorcerer's Stone (Harry Potter, #1) 

popularity:
    The Hunger Games (The Hunger Games, #1)
    To Kill a Mockingbird
    Twilight (Twilight, #1)
    Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)
    Harry Potter and the Chamber of Secrets (Harry Potter, #2)

svd:
    The Hunger Games (The Hunger Games, #1)
    Twilight (Twilight, #1)
    Harry Potter and the Chamber of Secrets (Harry Potter, #2)
    Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)
    The Lion, the Witch, and the Wardrobe (Chronicles of Narnia, #1)

svd_rating>=4:
    Harry Potter and the Chamber of Secrets (Harry Potter, #2)
    Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)
    The Little Prince
    Twilight (Twilight, #1)
    The Hobbit

content_tfidf_full:
    Harry Potter Boxset (Harry Potter, #1-7)
    Harry Potter Boxed Set, Books 1-5 (Harry Potter, #1-5)
    The Harry Potter Collection 1-4 (Harry Potter, #1-4)
    Harry Potter and the P